# 04 — Language Models (Module B)
**NewsBot Intelligence System 2.0 | ITAI 2373**

LLM integration via gemini
- B.1 Intelligent Summarization
- B.2 Content Enhancement
- B.3 Article Q&A
- B.4 Insight Generation

In [31]:
!pip install groq

In [32]:
import requests, json, time
from IPython.display import Markdown, display

# General LLM parameters (will be used by Gemini)
TEMPERATURE  = 0.3
MAX_TOKENS   = 512

In [33]:
from groq import Groq
from google.colab import userdata

# Retrieve API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Initialize the Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Define the default model to be used
GROQ_MODEL_NAME = 'llama-3.1-8b-instant' # Updated to a different, currently supported Groq model

print('Groq API configured and client initialized.')
print(f'Default Groq model set to: {GROQ_MODEL_NAME}. You can change this to another available Groq model if needed.')

Groq API configured and client initialized.
Default Groq model set to: llama-3.1-8b-instant. You can change this to another available Groq model if needed.


In [34]:
from google.colab import drive
drive.mount('/content/data')
PROJECT = '/content/data/MyDrive/ITAI2373-NewsBot-Final'

Drive already mounted at /content/data; to attempt to forcibly remount, call drive.mount("/content/data", force_remount=True).


In [35]:
# ── Verify df_final ───────────────────────────────────────────────
import os, pandas as pd
if os.path.exists(f'{PROJECT}/data/processed/df_final.pkl'):
    df_final = pd.read_pickle(f'{PROJECT}/data/processed/df_final.pkl')
    print(f'Loaded df_final: {df_final.shape}')
    print(f'Columns: {list(df_final.columns)}')
else:
    raise FileNotFoundError(
        'df_final.pkl not found.\n'
        'Run the previous notebook first and make sure it saved df_final.'
    )

Loaded df_final: (1490, 19)
Columns: ['ArticleId', 'Text', 'Category', 'text_length', 'word_count', 'doc_id', 'cleaned_text', 'entities', 'sentiment_compound', 'sentiment_label', 'passive_voice_instance_count', 'top_tfidf', 'lda_dominant_topic', 'lda_topic_confidence', 'lda_topic_label', 'nmf_dominant_topic', 'nmf_topic_confidence', 'nmf_topic_label', 'cluster']


In [36]:
# The groq_client object will be initialized in the next cell after API key setup.
# Make sure to run the cells in order for this to work.

def call_llm(prompt, system=None):
    """Call Groq API."""
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})

    response = groq_client.chat.completions.create(
        messages=messages,
        model=GROQ_MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return response.choices[0].message.content.strip()

def extract_json(text):
    """Parse JSON from LLM response, stripping markdown fences."""
    clean = text.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
    try:
        return json.loads(clean)
    except:
        return {'raw_response': text}

print('LLM helpers ready (for Groq).')

LLM helpers ready (for Groq).


## B.1 — Intelligent Summarization

In [37]:
def generate_summary(article_text, max_sentences=3, Category=None):
    system = ('You are a professional news editor. Write accurate concise summaries. ' + \
              'Preserve all named entities exactly as written.')
    cat_hint = f' This is a {Category} article.' if Category else ''
    prompt = (f'Summarize this news article in exactly {max_sentences} sentences.{cat_hint} ' + \
              f'Cover Who, What, When, Where, Why. Output only the summary, no preamble.\n\n' + \
              f'ARTICLE:\n{article_text[:3000]}')
    summary = call_llm(prompt, system)
    orig_words    = len(article_text.split())
    summary_words = len(summary.split())
    return {
        'summary':           summary,
        'word_count':        summary_words,
        'compression_ratio': round(1 - summary_words/max(orig_words,1), 3)
    }

# Demo on one article per category
print('B.1 — Summarization Demo (with Groq)\n' + '='*55)
for cat in ['tech','business','politics']:
    sample = df_final[df_final.Category==cat].sample(1, random_state=42).iloc[0]
    result = generate_summary(sample.Text, max_sentences=3, Category=cat)
    print(f'\n[{cat.upper()}]')
    print(f'Summary: {result["summary"]}')
    print(f'Words: {result["word_count"]} | Compression: {result["compression_ratio"]*100:.0f}%')

B.1 — Summarization Demo (with Groq)

[TECH]
Summary: The European Parliament's Legal Affairs Committee (JURI) has ordered a rewrite of the Computer Implemented Inventions Directive, a proposal to grant patent protection to computer-based inventions, after it failed to gain support from MEPs. The directive has been criticized by opponents who fear it could favor large firms over small ones and stifle open-source software innovation, while supporters argue it would allow companies to protect their inventions. The decision comes after Poland, a large EU member state, rejected the adoption of the directive twice in two months, amid intense lobbying and pressure from national parliaments.
Words: 95 | Compression: 67%

[BUSINESS]
Summary: Nasdaq Stock Market, the owner of the technology-dominated Nasdaq stock index, plans to sell $100m (£52m) worth of shares to the public and list itself on the market it operates. The share sale, which will not generate revenue for Nasdaq, is intended for i

## B.2 — Content Enhancement

In [43]:
def enhance_content(article_text, category=None, entities=None):
    entity_hint = ''
    if entities:
        all_ents = [e for lst in entities.values() for e in lst[:3]]
        if all_ents:
            entity_hint = f' Key entities: {chr(44).join(all_ents[:5])}.'
    system = ('You are a senior news analyst. '
              'Respond ONLY with valid JSON. No markdown, no preamble.')
    prompt = (f'Analyze this news article and return JSON with keys:\n'
              f'  background_context (2-3 sentences of relevant background)\n'
              f'  related_trends     (2-3 sentences on broader trends)\n'
              f'  implications       (2-3 sentences on potential impact)\n'
              f'  entities_to_watch  (list of 3-5 entity name strings)\n\n'
              f'Article:{" Category: "+category if category else ""}{entity_hint}\n{article_text[:2500]}')
    raw = call_llm(prompt, system)
    return extract_json(raw)

# Demo
sample = df_final[df_final.Category=='tech'].sample(1, random_state=1).iloc[0]
result = enhance_content(sample.Text, category='tech', entities=sample.get('entities',{}))
print('B.2 — Content Enhancement Demo')
print('='*55)
for key in ['background_context','related_trends','implications']:
    if key in result:
        print(f'\n{key.upper().replace("_"," ")}:')
        print(result[key])
if 'entities_to_watch' in result:
    print(f'\nENTITIES TO WATCH: {result["entities_to_watch"]}')


B.2 — Content Enhancement Demo

BACKGROUND CONTEXT:
['Microsoft is planning to enhance the security of its Windows and Internet Explorer platforms.', "The move comes as identity fraud is one of the UK's fastest-growing crimes, with an estimated £1.3bn lost last year.", "Microsoft's previous attempts at secure online transactions, such as Passport and Hailstorm, were criticized for storing user information centrally."]

RELATED TRENDS:
['The growing concern over online security and identity theft is driving companies to develop more secure solutions.', "The trend of storing sensitive information locally on a user's PC, rather than centrally, is becoming more popular.", 'The need for more user control over personal information is a key driver in the development of secure online systems.']

IMPLICATIONS:
['The new ID system could significantly reduce the risk of identity theft and online fraud.', "The system's focus on user control could lead to increased adoption and trust in online serv

## B.3 — Article Q&A (Multi-turn)

In [47]:
class ArticleQueryEngine:
    """Stateful multi-turn Q&A engine for a single article."""
    SYSTEM = ('You are NewsBot 2.0, an expert AI news analyst. '
              'Answer questions about news articles clearly and concisely. '
              'Ground responses in the provided article context.')

    def __init__(self, article_text, category=None):
        self.article_text = article_text
        self.category     = category
        self._history     = []
        cat_str = f'[{category.upper()}] ' if category else ''
        self._context = f'{self.SYSTEM}\n\nArticle: {cat_str}\n{article_text[:3000]}'

    def ask(self, question):
        history_str = ''
        if self._history:
            history_str = '\n'.join(
                f'Q: {t["question"]}\nA: {t["answer"]}' for t in self._history[-4:]
            )
            history_str = f'\nConversation history:\n{history_str}\n'
        prompt = f'{self._context}{history_str}\nQuestion: {question}\nAnswer:'
        answer = call_llm(prompt)
        self._history.append({'question':question,'answer':answer})
        return answer

    def show_history(self):
        for i,t in enumerate(self._history,1):
            print(f'[Q{i}] {t["question"]}')
            print(f'[A{i}] {t["answer"]}\n')

    def reset(self): self._history = []

# Demo — multi-turn conversation
print('B.3 — Multi-turn Q&A Demo')
print('='*55)
sample = df_final[df_final.Category=='politics'].sample(1, random_state=5).iloc[0]
print(f'Article preview: {sample.Text[:200]}...\n')

engine = ArticleQueryEngine(sample.Text, category='politics')
questions = [
    'What is the main policy being discussed?',
    'Who are the key people or parties involved?',
    'What are the potential implications of this?'
]
for q in questions:
    print(f'Q: {q}')
    a = engine.ask(q)
    print(f'A: {a}\n')


B.3 — Multi-turn Q&A Demo
Article preview: iraqis win death test case probe the family of an iraqi civilian allegedly killed by uk troops have won a challenge against the government s refusal to order a full inquiry.  the high court ruled on t...

Q: What is the main policy being discussed?
A: The main policy being discussed is the UK's jurisdiction and responsibility in investigating the deaths of Iraqi civilians allegedly killed by UK troops in Iraq, specifically in relation to the European Convention on Human Rights.

Q: Who are the key people or parties involved?
A: The key people or parties involved in this case are:

1. Baha Mousa, the Iraqi civilian who was allegedly killed by UK troops in 2003.
2. His family, who are seeking a full inquiry into his death.
3. Phil Shiner, the solicitor representing the Mousa family.
4. Lord Justice Rix and Mr. Justice Forbes, the judges who ruled in favor of the Mousa family's challenge against the government's refusal to order a full inquiry.
5

## B.4 — Insight Generation

In [49]:
def generate_insights(article_text, nlp_metadata=None):
    meta_str = ''
    if nlp_metadata:
        parts = []
        if 'sentiment_label' in nlp_metadata:
            parts.append(f'Sentiment: {nlp_metadata["sentiment_label"]} ({nlp_metadata.get("sentiment_compound","N/A")})')
        if 'top_tfidf' in nlp_metadata:
            terms = [t for t,_ in nlp_metadata['top_tfidf'][:5]]
            parts.append(f'Top TF-IDF: {chr(44).join(terms)}')
        meta_str = '\n'.join(parts)

    system = ('You are an expert NLP analyst. '
              'Respond ONLY with valid JSON. Ground all observations in the article.')
    prompt = (f'Analyze this article and return JSON with keys:\n'
              f'  key_findings         (list of 3-5 finding strings)\n'
              f'  patterns             (list of 2-3 pattern strings)\n'
              f'  entities_of_interest (list of entity name strings)\n'
              f'  sentiment_drivers    (list of 2-3 phrases driving sentiment)\n'
              f'  recommended_queries  (list of 4 natural language questions)\n\n'
              f'NLP metadata:\n{meta_str}\n\nArticle:\n{article_text[:2500]}')
    raw = call_llm(prompt, system)
    return extract_json(raw)

# Demo
sample = df_final[df_final.Category=='business'].sample(1, random_state=10).iloc[0]
meta = {
    'sentiment_label':    sample.get('sentiment_label',''),
    'sentiment_compound': sample.get('sentiment_compound',''),
    'top_tfidf':          sample.get('top_tfidf',[])
}
result = generate_insights(sample.Text, nlp_metadata=meta)

print('B.4 — Insight Generation Demo')
print('='*55)
for key in ['key_findings','patterns','sentiment_drivers','recommended_queries']:
    if key in result:
        print(f'\n{key.upper().replace("_"," ")}:')
        items = result[key] if isinstance(result[key], list) else [result[key]]
        for item in items:
            print(f'  - {item}')


B.4 — Insight Generation Demo

KEY FINDINGS:
  - US Bank loses customer details, affecting 1.2 million federal employees
  - Missing tapes may have been stolen from a plane by baggage handlers
  - Bank officials claim no evidence of criminal activity, but the Secret Service is investigating

PATTERNS:
  - loss of customer data due to theft or mismanagement
  - investigation by law enforcement agencies
  - concerns about background checks for sensitive positions

SENTIMENT DRIVERS:
  - vulnerability to identity theft
  - lack of transparency from the bank
  - concerns about security and background checks

RECOMMENDED QUERIES:
  - What are the consequences of losing customer data for US federal employees?
  - How does the bank plan to prevent similar incidents in the future?
  - What are the current regulations for background checks for baggage handlers?
  - What is the role of the Secret Service in investigating financial crimes?


## B.5 — Batch Pipeline

In [53]:
def run_module_b(df, n_samples=5, random_state=42):
    """Run full Module B pipeline on a sample of articles."""
    sample = df.groupby('Category').apply(
        lambda g: g.sample(min(1, len(g)), random_state=random_state)
    ).reset_index(drop=True)
    if n_samples:
        sample = sample.head(n_samples)

    results = []
    for _, row in sample.iterrows():
        print(f'Processing [{row.Category}]...')
        entry = {'doc_id': row.get('doc_id', _), 'category': row.Category}
        try:
            entry['summary'] = generate_summary(row.text, category=row.Category)
        except Exception as e:
            entry['summary'] = {'error': str(e)}
        try:
            meta = {'sentiment_label': row.get('sentiment_label',''),
                    'sentiment_compound': row.get('sentiment_compound',0),
                    'top_tfidf': row.get('top_tfidf',[])}
            entry['insights'] = generate_insights(row.Text, nlp_metadata=meta)
        except Exception as e:
            entry['insights'] = {'error': str(e)}
        results.append(entry)
        time.sleep(0.5)

    return results

print('Running Module B batch pipeline (1 article per category)...')
batch_results = run_module_b(df_final, n_samples=5)
print(f'\nProcessed {len(batch_results)} articles.')
for r in batch_results:
    s = r.get('summary',{})
    print(f'  [{r["category"]}] Summary: {s.get("word_count","?")} words, compression {s.get("compression_ratio","?")}')


Running Module B batch pipeline (1 article per category)...
Processing [business]...


/tmp/ipykernel_31550/974259102.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample = df.groupby('Category').apply(


Processing [entertainment]...
Processing [politics]...
Processing [sport]...
Processing [tech]...

Processed 5 articles.
  [business] Summary: ? words, compression ?
  [entertainment] Summary: ? words, compression ?
  [politics] Summary: ? words, compression ?
  [sport] Summary: ? words, compression ?
  [tech] Summary: ? words, compression ?


In [54]:
# ── Save df_final for next notebook ──────────────────────────────
df_final.to_pickle(f'{PROJECT}/data/processed/df_final.pkl')
print(f'Saved df_final: {df_final.shape} → {PROJECT}/data/processed/df_final.pkl')
print(f'Columns: {list(df_final.columns)}')

Saved df_final: (1490, 19) → /content/data/MyDrive/ITAI2373-NewsBot-Final/data/processed/df_final.pkl
Columns: ['ArticleId', 'Text', 'Category', 'text_length', 'word_count', 'doc_id', 'cleaned_text', 'entities', 'sentiment_compound', 'sentiment_label', 'passive_voice_instance_count', 'top_tfidf', 'lda_dominant_topic', 'lda_topic_confidence', 'lda_topic_label', 'nmf_dominant_topic', 'nmf_topic_confidence', 'nmf_topic_label', 'cluster']
